<div dir="rtl">

# الدرس 43: من النص إلى أرقام: التقطيع

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### الخوارزمية من الصفر

</div>

In [ ]:
from collections import Counter

def train_bpe(word_counts, n_merges):
    """word_counts: {word: count}. Returns the merge rules and the final segmentation."""
    words = {w: list(w) for w in word_counts}              # start from characters
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for w, toks in words.items():
            for a, b in zip(toks, toks[1:]):
                pairs[(a, b)] += word_counts[w]           # weighted by word frequency
        if not pairs:
            break
        best = max(pairs, key=pairs.get)                   # ties: the first pair seen wins
        merges.append(best)
        for w in words:
            words[w] = apply_merge(words[w], best)
    return merges, words

def apply_merge(toks, pair):
    out, i = [], 0
    while i < len(toks):
        if i + 1 < len(toks) and (toks[i], toks[i + 1]) == pair:
            out.append(toks[i] + toks[i + 1])
            i += 2
        else:
            out.append(toks[i])
            i += 1
    return out

def encode_word(word, merges):
    toks = list(word)
    for pair in merges:                                    # same order as training
        toks = apply_merge(toks, pair)
    return toks

corpus = {"كتب": 8, "كتاب": 4, "مكتب": 6, "مكتبة": 2, "يكتب": 5}
merges, words = train_bpe(corpus, 3)
print("merges:", merges)
print("words :", words)
for w in ["يكتبون", "مكتوب", "كاتب", "فكتب"]:
    print(w, "->", encode_word(w, merges))

<div dir="rtl">

### الحروف والبايتات

</div>

In [ ]:
for s in ["cat", "سلام", "😀"]:
    b = s.encode("utf-8")
    print(f"{s!r}: {len(s)} characters, {len(b)} bytes -> {list(b)}")

<div dir="rtl">

### نص حقيقي: أعمال شكسبير

</div>

In [ ]:
import os
import re
import urllib.request

URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(URL, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", encoding="utf-8").read()
print(f"{len(text):,} characters")
print(text[:120])

In [ ]:
word_counts = Counter(re.findall(r"\w+|[^\w\s]", text.lower()))
print(f"{len(word_counts):,} different words, {sum(word_counts.values()):,} words in total")
merges, _ = train_bpe(dict(word_counts), 30)
print([a + b for a, b in merges])

<div dir="rtl">

### مقطّع حقيقي على البايتات

</div>

In [ ]:
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=1000, special_tokens=["<|endoftext|>"],
                              initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                              show_progress=False)
tokenizer.train_from_iterator([text], trainer)
print("vocabulary size:", tokenizer.get_vocab_size())

for sentence in ["To be, or not to be: that is the question.",
                 "Artificial intelligence is changing the world.",
                 "الذكاء الاصطناعي يغيّر العالم"]:
    enc = tokenizer.encode(sentence)
    print(f"\n{len(sentence)} characters, {len(sentence.encode())} bytes -> {len(enc.ids)} tokens")
    print(enc.tokens[:16])
    print("decoded back:", tokenizer.decode(enc.ids))

<div dir="rtl">

## تمرين: دمجات بيدك، على مجموعة جديدة

مجموعة كلمات إنجليزية، مع مرات ظهورها:

| الكلمة | low | lower | newest | widest |
|---|---|---|---|---|
| مرات الظهور | 5 | 2 | 6 | 3 |

1. نفّذ ثلاث دمجات بيدك. عند التعادل، اختر الزوج الذي يظهر أولاً وأنت تقرأ الكلمات بالترتيب، من اليسار لليمين.
2. ما تقطيع الكلمات الأربع بعد الدمجات الثلاث؟
3. تحقق بالكود، وفي الأداة التفاعلية باختيار المجموعة الثانية.

</div>

In [ ]:
# اكتب حلّك هنا
